# Scenarios

One way of modeling uncertainty in decision making is using _scenarios_. A scenario in optimization context is typically a set of parameters or constraints that describes what our optimization problem might look like in a plausible future.

In DESDEO scenarios are described by a [ScenarioModel][desdeo.problem.scenario.ScenarioModel]. It contains base_problem that is a [Problem][desdeo.problem.schema.Problem] object. The base_problem is then modified by the different [Scenario][desdeo.problem.scenario.Scenario]s which are stored in a dict called scenarios. How different scenarios flow into one another is described by scenario_tree dict. The scenarios can also be assigned probabilities, which are stored in the scenario_probabilities dict.

## Building a ScenarioModel

We will take a look at how a [ScenarioModel][desdeo.problem.scenario.ScenarioModel] is constructed with the help of an example. The first thing our ScenarioModel needs is a base_problem that will be modified by the different scenario. 

We will use the [summer_cabin_electricity][desdeo.problem.testproblems.scummer_cabin_electricity] Problem as our base_problem. It is a MILP-problem with a few thousand variables. We will use the split version of the problem, where the decision variables relating to electricity usage have been split into three time periods. The reason for this will become apparent later.

In [8]:
from pprint import pprint

from desdeo.problem.testproblems import summer_cabin_battery_problem_split

base = summer_cabin_battery_problem_split(initial_soc=0, n_panels_max=50)
pprint({i: base.objectives[i].name for i in range(len(base.objectives))})


{0: 'Total electricity cost', 1: 'Total investment cost'}


The problem is about choosing what kind of investments should be done in to the electricity supply of the summer cabin. The options are installing a battery with a capacity of 14-42 kWh or installing a number of solar panels that produce up to 160W of power each depending on the weather and the time of day.

### Defining scenarios

The type of uncertainty we want to introduce is the possibility of losing the connection to the power grid because of a storm. We have two storms that each may or may not cause an electricity outage for 4 hours.

To describe the scenarios arising from these possible events, we write a scenario_tree and put the scenario_probabilities in a dict.

In [9]:
scenario_tree={
    "ROOT": ["S1", "S2"],
    "S1": ["S1a", "S1b"],
    "S2": ["S2a", "S2b"],
    "S1a": [],
    "S1b": [],
    "S2a": [],
    "S2b": [],
}
scenario_probabilities={
    "S1": 0.9,
    "S2": 0.1,
    "S1a": 0.81,
    "S1b": 0.09,
    "S2a": 0.09,
    "S2b": 0.01,
}

# Including the leaf nodes in the scenario tree is optional,
# so we can also define the scenario tree as follows:
scenario_tree={
    "ROOT": ["S1", "S2"],
    "S1": ["S1a", "S1b"],
    "S2": ["S2a", "S2b"],
}

The `scenario_tree` is a dict, where the keys represent scenarios, and the values are lists that show which scenarios follow them. Leaf scenarios have an empty list, because there is no scenario following them. You can leave the leaf scenarios out of your scenario tree keys if you want.

The tree above shows that first either scenario `S1` or `S2` happens. They can then be followed by second stage scenarios `S1a` or `S1b` and `S2a` or `S2b` respectively.

If all of your scenarios would happen at the same time (from the perspective of the mathematical model), you would only have the `ROOT` scenario followed by a list of your scenarios. In that case, you could just provide the list of scenario names to the ScenarioModel instead a dict containing the tree.

The `scenario_probabilities` dict assigns a probability to every scenario in the `scenario_tree`. The ScenarioModel validates that the probabilities of child scenarios sum up to the probability of their parent.

### Describing the effects of the scenarios

We also need to describe how the scenarios change our `base_problem`. [ScenarioModel][desdeo.problem.scenario.ScenarioModel] handles this by having lists of constants, variables, objectives, constraints, extra functions, and scalarization functions, which are then assigned to scenarios as need be. It works like this, because it is often desirable to use, for example, same constraints in multiple different scenarios, and having them all be drawn from a single lists saves space is computer memory and database.

We wanted to describe an electricity outage, so let's define constraints that say we cannot import or export electricity at specified times.

In [ ]:
from desdeo.problem import Constraint, ConstraintTypeEnum

con_pool: list[Constraint] = []

H = 4 # hours of grid outage
for t in range(1, H + 1):
    con_pool.append(
        Constraint(
            name=f"Outage no-trade s2 t={t}",
            symbol=f"outage_trade_s2_{t}",
            func=["Add", ["At", "buy_s2", t], ["At", "sell_s2", t]],
            cons_type=ConstraintTypeEnum.EQ,
            is_linear=True,
            is_convex=True,
            is_twice_differentiable=True,
        )
    )